In [1]:
import matplotlib.pylab as plt
import numpy as np
import numpy.random as rgn
import pickle

from tvb.simulator.lab import *
from tvb.simulator.lab import cortex as ct
from tvb.simulator.lab import region_mapping as rm
from tvb.simulator.lab import connectivity as cnc

In [6]:
values = [5,15,25,35,45,55,65,75,85,95,105,110,115,120,125,130,135,140,145,150]
for adap_values in values:

    n_areas = 68
    mu, sigma = 0, 1 # mean and standard deviation
    s = np.random.normal(mu, sigma, n_areas)

    adap_fixed = adap_values
    adaptation = np.ones(n_areas)*adap_fixed + (0.1*adap_fixed)*s

    # execute a cell with {adex} to show equations
    adex = models.ZerlautAdaptationSecondOrder(
        g_L = np.r_[10.0],
        E_L_e = np.r_[-64.0], 
        E_L_i = np.r_[-65.0],
        C_m = np.r_[200.0],
        b_e = adaptation,     
        a_e = np.r_[0.0],
        b_i = np.r_[0.0],
        a_i = np.r_[0.0],
        tau_w_e = np.r_[500.0],
        tau_w_i = np.r_[1.0],
        E_e = np.r_[0.0],
        E_i = np.r_[-80.0],
        Q_e = np.r_[1.5],
        Q_i = np.r_[5.0],
        tau_e = np.r_[5.0],
        tau_i = np.r_[5.0],
        N_tot = np.r_[10000],
        p_connect_e = np.r_[0.05], 
        p_connect_i = np.r_[0.05], 
        g = np.r_[0.2],
        T = np.r_[20.0],
        P_e = np.r_[
            [
                -0.05017034,
                0.00451531,
                -0.00794377,
                -0.00208418,
                -0.00054697,
                0.00341614,
                -0.01156433,
                0.00194753,
                0.00274079,
                -0.01066769,
            ]
        ],    
        P_i = np.r_[
            [
                -0.05184978,
                0.0061593,
                -0.01403522,
                0.00166511,
                -0.0020559,
                0.00318432,
                -0.03112775,
                0.00656668,
                0.00171829,
                -0.04516385,
            ]
        ],
        external_input_ex_ex = np.r_[0.315 * 1e-3],
        external_input_ex_in = np.r_[0.000],
        external_input_in_ex = np.r_[0.315 * 1e-3],
        external_input_in_in = np.r_[0.000],
        K_ext_e = np.r_[400],
        K_ext_i = np.r_[0],
        tau_OU = np.r_[5.0],
        weight_noise = np.r_[3e-4],
    )
    adex.variables_of_interest = ['E', 'I', 'C_ee', 'C_ei', 'C_ii', 'W_e', 'W_i', 'ou_drift']

    adex.state_variable_range["E"] = [0.000, 0.000]
    adex.state_variable_range["I"] = [0.00, 0.00]
    adex.state_variable_range["C_ee"] = [0.0, 0.0]
    adex.state_variable_range["C_ei"] = [0.0, 0.0]
    adex.state_variable_range["C_ii"] = [0.0, 0.0]
    adex.state_variable_range["W_e"] = [100., 100.0] # 100?
    adex.state_variable_range["W_i"] = [0.0, 0.0]
    adex.state_variable_range["ou_drift"] = [0.0, 0.0]

    # Set connectivity
    path_c = "C:\\Users\\..."

    conn = connectivity.Connectivity.from_file(
        os.path.abspath(path_c + "Connectivity.zip"),
    )
    conn.weights = conn.weights/(np.sum(conn.weights,axis=0)+1e-12)
    conn.speed = np.r_[4.0]


    # Set simulator - uncomment stimulus if wanted
    sim = simulator.Simulator(
        model=adex,
        connectivity=conn,
        conduction_speed=conn.speed.item(), 
        coupling=coupling.Linear(a = np.r_[0.3], b = np.r_[.0]),
        integrator=integrators.HeunStochastic(
            noise = noise.Additive(nsig=np.r_[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], noise_seed = 10), #variables ['E', 'I', 'C_ee', 'C_ei', 'C_ii', 'W_e', 'W_i', 'ou_drift']
            dt = 0.1,
        ),
        monitors=[monitors.TemporalAverage(period=1.0)]
    ).configure()

    transient = 2000
    time_simulation = 62000

    (out_t, out_d), = sim.run(simulation_length = time_simulation + transient)

    # cut transient
    out_d = out_d[transient:,:,:,:]
    out_t = out_t[transient:]

    data_ = {}
    data_["Time"] = out_t*0.001
    data_["TimeSeries_chs"] = out_d[:,0,:,0]*1e3

    with open("DataModel_"+str(adap_fixed)+".pkl", "wb") as f:
        pickle.dump(data_, f)